In [ ]:
import os
import numpy as np
import pandas as pd

df = pd.read_csv("~/Desktop/NLP_Submission/Data/2_Merged_data/youtube_merged/youtube_merged.csv")     

TARGET = {
    "Spanish":     120_212,
    "English":     854_278,
    "Russian":     110_473,
    "German":      66_284,
    "Portuguese":  73_697,
    "French":      69_191,
    "Japanese":    69_905,
    "Chinese":      21_513,
    "Italian":      16_716,
    "Polish":       11_192,
    "Thai":         7_558,
    "Turkish":      5_087,
    "Ukrainian":    5_233,
    "Arabic":       7_268,
}

MIN_STREAMERS = 30        
RANDOM_STATE  = 42        

In [ ]:
df.info()

In [ ]:
def missing_values(df):
    missing_values = df.isna().sum()
    
    # Calculate the percentage of missing values per column
    missing_percentage = (missing_values / len(df)) * 100
    
    # Combine the total and percentage into a DataFrame for better presentation
    missing_data = pd.DataFrame({
        'Missing Values': missing_values,
        'Percentage': missing_percentage
    })
    
    # Display the missing data
    return missing_data

In [ ]:
missing_values(df)

In [ ]:
def downsample_language(sub_df: pd.DataFrame, target: int) -> pd.DataFrame:
    if len(sub_df) <= target:
        return sub_df

    g = sub_df.groupby("streamer_name", group_keys=False)
    streamer_sizes = g.size()
    tot_messages   = streamer_sizes.sum()

    keep_streamers = streamer_sizes.index
    if len(keep_streamers) > MIN_STREAMERS:
        keep_streamers = keep_streamers

    alloc = (
        (streamer_sizes[keep_streamers] / tot_messages * target)
        .apply(lambda x: max(1, int(round(x))))
    )

    overflow = alloc.sum() - target
    if overflow > 0:
        candidates = alloc[alloc > 1].index.to_list()
        np.random.seed(RANDOM_STATE)
        for i in range(overflow):
            s = np.random.choice(candidates)
            alloc[s] -= 1
            if alloc[s] == 1:   
                candidates.remove(s)

    sampled_parts = []
    for s, n in alloc.items():
        part = sub_df[sub_df["streamer_name"] == s].sample(
            n=n, random_state=RANDOM_STATE, replace=False
        )
        sampled_parts.append(part)

    return pd.concat(sampled_parts, ignore_index=True)

sampled_frames = []
for lang, target in TARGET.items():
    lang_df = df[df["language"] == lang]
    sampled_frames.append(downsample_language(lang_df, target))

df_reduced = pd.concat(sampled_frames, ignore_index=True)
print(df_reduced["language"].value_counts().sort_values(ascending=False))
print("Total rows:", len(df_reduced))   

In [ ]:
streamer_counts = (
    df_reduced
        .groupby("language")["streamer_name"]
        .nunique()                           
        .rename("n_streamers")
        .sort_values(ascending=False)       
)
print(streamer_counts)

In [ ]:
OUTFILE = os.path.expanduser("~/Desktop/NLP_Submission/Data/3_Scaled_data/twitch_scaled.csv")
df_reduced.to_csv(OUTFILE, index=False)

print("Saved:", OUTFILE)
print("\nDistinct streamers per language:")
print(
    df_reduced.groupby("language")["streamer_name"]
              .nunique()
              .sort_values(ascending=False)
)
print("\nTotal rows:", len(df_reduced))

In [ ]:
import os
import numpy as np
import pandas as pd
df = pd.read_csv("~/Desktop/NLP_Submission/Data/2_Merged_data/twitch_merged/twitch_merged.csv") 
OUTFILE = os.path.expanduser("~/Desktop/NLP_Submission/Data/3_Scaled_data/twitch_scaled.csv")


TARGET_MSG = {               
    "Spanish":     240_425,
    "English":   1_708_556,
    "Russian":     220_947,
    "German":      132_568,
    "Portuguese":  147_395,
    "French":      138_382,
    "Japanese":    121_811,
    "Chinese":      43_026,
    "Italian":      33_433,
    "Polish":       22_385,
    "Thai":         15_117,
    "Turkish":      10_175,
    "Ukrainian":    10_466,     
    "Arabic":       14_536,
}
MIN_STREAMERS = 30
RANDOM_STATE  = 42

def downsample_language(sub: pd.DataFrame, target_msgs: int) -> pd.DataFrame:
    orig_msgs      = len(sub)
    orig_streamers = sub["streamer_name"].nunique()

    if orig_msgs <= target_msgs:
        return sub

    scale              = target_msgs / orig_msgs
    target_streamers   = max(MIN_STREAMERS, int(round(orig_streamers * scale)))
    target_streamers   = min(target_streamers, orig_streamers)

    stream_counts = sub["streamer_name"].value_counts()

    chosen = stream_counts.index[:target_streamers].tolist()
    k = target_streamers
    while stream_counts[chosen].sum() < target_msgs:
        k += 1
        chosen = stream_counts.index[:k].tolist()

    sub_chosen = sub[sub["streamer_name"].isin(chosen)]

    counts_chosen = sub_chosen["streamer_name"].value_counts()
    alloc = (
        (counts_chosen / counts_chosen.sum() * target_msgs)
        .round()
        .astype(int)
        .clip(lower=1)               
    )

    diff = alloc.sum() - target_msgs
    if diff != 0:
        np.random.seed(RANDOM_STATE)
        sign = 1 if diff < 0 else -1   
        diff = abs(diff)
        cand = alloc.index.tolist()
        while diff:
            s = np.random.choice(cand)
            if alloc[s] > 1 or sign == 1:
                alloc[s] += sign
                diff    -= 1

    parts = [
        sub_chosen[sub_chosen["streamer_name"] == s]
            .sample(n=n, replace=False, random_state=RANDOM_STATE)
        for s, n in alloc.items()
    ]
    return pd.concat(parts, ignore_index=True)

scaled_parts = []
for lang, target in TARGET_MSG.items():
    lang_df = df[df["language"] == lang]
    scaled_parts.append(downsample_language(lang_df, target))

df_scaled = pd.concat(scaled_parts, ignore_index=True)
df_scaled.to_csv(OUTFILE, index=False)

print(f"File salvato in: {OUTFILE}")
print("\nConteggio streamer per lingua:")
print(
    df_scaled.groupby("language")["streamer_name"]
             .nunique()
             .sort_values(ascending=False)
)
print("\nTotale righe:", len(df_scaled))